# Main multimodal experiments

Standalone Kaggle/Colab notebook for the text-only, image-only, ViT concatenation, ViT cross-attention, and optional CLIP experiments. Set `SMOKE_TEST=False` only after the smoke run succeeds. Attach or upload the repository so it contains `src/`, `configs/`, and `data/splits/`.

In [ ]:
%pip -q install -U torch transformers accelerate pandas numpy scikit-learn pillow pyyaml tqdm matplotlib seaborn

In [ ]:
import os
from pathlib import Path
import sys
import torch

SMOKE_TEST = True
SMOKE_SIZE = 32
REPO_ROOT_OVERRIDE = ""

candidates = [Path(REPO_ROOT_OVERRIDE)] if REPO_ROOT_OVERRIDE else []
candidates += [Path("/kaggle/working/FakeNewsDetector"), Path("/content/FakeNewsDetector"), Path.cwd()]
for base in [Path("/kaggle/input"), Path("/content")]:
    if base.exists():
        candidates += [p.parent.parent.parent for p in base.rglob("run_experiment.py")]
REPO_ROOT = next((p for p in candidates if (p / "src/experiments/run_experiment.py").exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Upload/clone FakeNewsDetector and set REPO_ROOT_OVERRIDE to its root.")
os.environ["PYTHONPATH"] = str(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
OUTPUT_ROOT = Path("/kaggle/working/research_outputs" if Path("/kaggle").exists() else "/content/research_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_ROOT)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")
print("Smoke test:", SMOKE_TEST)

In [ ]:
import copy
import yaml
from src.experiments.run_experiment import run

CONFIGS = [
    "text_roberta.yaml",
    "image_vit.yaml",
    "concat_vit.yaml",
    "cross_attention_vit.yaml",
    "cross_attention_clip.yaml",
]
results = []
for filename in CONFIGS:
    source = REPO_ROOT / "configs/experiments" / filename
    if not source.exists():
        print("Missing config, skipped:", source)
        continue
    with source.open(encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    config["experiment"]["output_dir"] = str(OUTPUT_ROOT)
    config["experiment"]["name"] = Path(filename).stem + ("_smoke" if SMOKE_TEST else "")
    config["experiment"]["smoke_test"] = SMOKE_TEST
    config["experiment"]["smoke_size"] = SMOKE_SIZE
    temp_config = OUTPUT_ROOT / (Path(filename).stem + "_runtime.yaml")
    temp_config.write_text(yaml.safe_dump(config), encoding="utf-8")
    try:
        results.append(run(str(temp_config)))
        print("Completed:", filename)
    except Exception as exc:
        print("Skipped/failed:", filename, type(exc).__name__, exc)

import pandas as pd
rows = []
for result in results:
    row = {"experiment": result["experiment"], **result["test_metrics"]}
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison.to_csv(OUTPUT_ROOT / "main_experiments.csv", index=False)
comparison

For the full experiment, set `SMOKE_TEST=False`, rerun the configuration cell and this execution cell. The test split is used only after validation macro-F1 checkpoint selection. CLIP may require more memory and can be skipped if its checkpoint is unavailable.

# Visual outputs

The following cell creates model-comparison bars, training curves, and confusion-matrix heatmaps from the executed run metadata.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
